# Data (re)Preprocessing — Treating outside London as an additional MSOA, National deprivation frame.

**Changes from the 20260615 version:**
- This version of data preprocessing extend flows with a "+1" MSOA for all outside London MSOAs. These flows treat London as either origin or destination.

**Everything else is identical to the 20260615 version.** 

**Purpose:**

The main analysis filters to London-internal flows only (both origin AND destination in London). This script extends the pipeline by also including flows where ONE endpoint is in London and the other is outside, treating the entire non-London area as a single synthetic "+1" MSOA.

However, previous preprocessing logic ranked MSOAs only within London. If we introduce outside London MSOAs, there is issue with their rankings. So in this file, all ~6,800 England MSOAs are ranked and assigned to deciles based on their IMD 2010 scores on a **national frame**.

Non-London areas are then collapsed into a single sythetic MSOA (`EXT_OUTSIDE`) whose decile is the population-weighted average of all non-London MSOAs' nationally-assigned deciles. This allows external flows to be classified as "wealthier inflow" or "poorer outflow" **within a consistent national hierarchy**, **testing whether including London-external flows changes the observed cascade-counter balance**.

**Design decisions:**

| Main analysis (existing) | This extension |
| :--- | :--- |
| Deciles: 983 London MSOAs only | Deciles: ~6,800 England MSOAs |
| Flows: London ↔ London | Flows: London ↔ anywhere |
| Frame: London-relative | Frame: National-relative |
| Purpose: Core analysis | Purpose: Sensitivity/extension |

**Input:**

census_od_2021_msoa.csv        (2021 MSOA-level OD, all E&W)
census_od_2011_oa.csv          (2011 OA-level OD, all E&W)
NSPCL_NOV22_UK_LU.csv          (postcode lookup: OA → LSOA → MSOA)
msoa_2011_to_2021_lookup.csv   (MSOA 2011 ↔ 2021 correspondence)
imd_2010.xls                   (IMD 2010 LSOA scores)
imd_2019.csv                   (IMD 2019, for mid-2015 LSOA populations)
ks101ew_lsoa_2011.csv          (2011 Census population by LSOA)
msoa_cascade_features_enriched_20260616.csv  (existing results to merge)

**Design:**
1. Aggregate IMD 2010 to all England MSOAs
2. Assign wealth deciles on the **national** distribution
3. Include London-external flows in the cascade computation
4. Compare results against the London-relative baseline


**Output:**

msoa_cascade_national_frame_20260622.csv

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import date
from scipy import stats
from pyprojroot import here

In [2]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — update paths to match your local setup
# ══════════════════════════════════════════════════════════════════════

ROOT = here()

DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Input files (same as main preprocessing)
imd_2010_path       = DATA_DIR / 'imd_2010.xls'
census_od_2021_path = DATA_DIR / 'census_od_2021_msoa.csv'
census_od_2011_path = DATA_DIR / 'census_od_2011_oa.csv'
lookup_path         = DATA_DIR / 'NSPCL_NOV22_UK_LU.csv'
msoa_lookup_path    = DATA_DIR / 'msoa_2011_to_2021_lookup.csv'
existing_path       = OUTPUT_DIR / 'msoa_cascade_features_enriched_20260616.csv'

# NEW: all-England KS101 population file
ks101_allengland_path = DATA_DIR / 'ks101ew_lsoa_2011_allengland.csv'

# Synthetic code for all non-London areas
EXTERNAL_CODE = 'EXT_OUTSIDE'

# IMD 2010 column names
IMD_2010_LSOA_COL  = 'LSOA CODE'
IMD_2010_SCORE_COL = 'IMD SCORE'

## 1. Load raw data

In [3]:
# ---- IMD 2010 (LSOA level, all England) ----
imd_2010 = pd.read_excel(imd_2010_path, sheet_name='IMD 2010')
imd_2010.columns = imd_2010.columns.str.strip()
print(f'IMD 2010: {len(imd_2010)} LSOAs')

# ---- KS101EW population (all England & Wales LSOAs) ----
ks101 = pd.read_csv(ks101_allengland_path)
# Detect LSOA code column: find column where values match E01/W01 pattern
_ks_code_col = next(
    c for c in ks101.columns
    if ks101[c].astype(str).str.match(r'^[EW]01\d{6}$').mean() > 0.9
)
_ks_pop_col = next(c for c in ks101.columns if 'all usual residents' in c.lower())
lsoa_pop = (ks101[[_ks_code_col, _ks_pop_col]]
            .rename(columns={_ks_code_col: 'lsoa11cd', _ks_pop_col: 'pop'}))
lsoa_pop['pop'] = pd.to_numeric(lsoa_pop['pop'], errors='coerce').fillna(0)
print(f'KS101EW: {len(lsoa_pop)} LSOAs, total pop = {lsoa_pop["pop"].sum():,.0f}')

# ---- NSPCL postcode lookup (LSOA → MSOA mapping) ----
nspcl = pd.read_csv(lookup_path, encoding='ISO-8859-1', low_memory=False)
lsoa_to_msoa = nspcl[['lsoa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa = nspcl[['oa11cd', 'msoa11cd']].drop_duplicates().dropna()
oa_to_msoa_dict = dict(zip(oa_to_msoa['oa11cd'], oa_to_msoa['msoa11cd']))
print(f'LSOA→MSOA: {len(lsoa_to_msoa)} | OA→MSOA: {len(oa_to_msoa_dict):,}')

# ---- MSOA 2021→2011 correspondence ----
msoa_11_21 = pd.read_csv(msoa_lookup_path)
msoa_11_21.columns = msoa_11_21.columns.str.strip().str.lower()
unchanged = msoa_11_21[msoa_11_21['msoa11cd'] == msoa_11_21['msoa21cd']]
msoa21_to_11 = dict(zip(unchanged['msoa21cd'], unchanged['msoa11cd']))
print(f'MSOA 2021→2011 (1:1 only): {len(msoa21_to_11)}')

# ---- Existing London-only results (for comparison at the end) ----
existing = pd.read_csv(existing_path)
london_msoas = set(existing['msoa11cd'].unique())
print(f'London analysis MSOAs: {len(london_msoas)}')

IMD 2010: 32482 LSOAs
KS101EW: 34753 LSOAs, total pop = 56,075,912
LSOA→MSOA: 42621 | OA→MSOA: 232,044
MSOA 2021→2011 (1:1 only): 7080
London analysis MSOAs: 983


## 2. National IMD aggregation: LSOA -> MSOA (all England)

Aggregate IMD 2010 to ALL England MSOAs, then qcut into deciles

Same logic as the main preprocessing:
- Population-weighted mean of LSOA IMD scores per MSOA
- Re-rank MSOAs on the aggregated scores
- Assign deciles via qcut on the national distribution

In [4]:
# ---- 2a. Join IMD scores → MSOA geography → population weights ----
imd_lsoa = imd_2010[[IMD_2010_LSOA_COL, IMD_2010_SCORE_COL]].rename(
    columns={IMD_2010_LSOA_COL: 'lsoa11cd', IMD_2010_SCORE_COL: 'imd_score'})

# LSOA → MSOA
imd_with_msoa = pd.merge(imd_lsoa, lsoa_to_msoa, on='lsoa11cd', how='inner')
print(f'IMD LSOAs matched to MSOA: {len(imd_with_msoa)} / {len(imd_lsoa)}')

# Add population weights
imd_with_msoa = pd.merge(imd_with_msoa, lsoa_pop, on='lsoa11cd', how='left')
imd_with_msoa['pop'] = imd_with_msoa['pop'].fillna(0)

IMD LSOAs matched to MSOA: 31672 / 32482


In [5]:
# ---- 2b. Population-weighted mean per MSOA ----
# (Same fallback as main notebook: if all weights are zero, use simple mean)
def pop_weighted_mean(group):
    total_pop = group['pop'].sum()
    if total_pop > 0:
        return np.average(group['imd_score'], weights=group['pop'])
    else:
        return group['imd_score'].mean()

msoa_imd = (imd_with_msoa
            .groupby('msoa11cd')
            .apply(pop_weighted_mean, include_groups=False)
            .reset_index(name='IMD_2010_national'))

print(f'MSOAs with aggregated IMD: {len(msoa_imd)}')
print(f'  London MSOAs:     {msoa_imd["msoa11cd"].isin(london_msoas).sum()}')
print(f'  Non-London MSOAs: {(~msoa_imd["msoa11cd"].isin(london_msoas)).sum()}')


MSOAs with aggregated IMD: 6778
  London MSOAs:     983
  Non-London MSOAs: 5795


In [6]:
# ---- 2c. Assign national wealth deciles ----
# qcut splits into 10 bins by score. Higher IMD score = more deprived.
# pd.qcut with labels=False gives bin 0 = lowest scores = LEAST deprived.
# We want Decile 1 = most deprived, so: Decile = 11 - (bin + 1)
msoa_imd['Wealth_Decile_National'] = (
    11 - (pd.qcut(msoa_imd['IMD_2010_national'], 10, labels=False) + 1)
)

# Verify direction: D1 should have highest IMD scores (most deprived)
d1_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 1,
                         'IMD_2010_national'].mean()
d10_score = msoa_imd.loc[msoa_imd['Wealth_Decile_National'] == 10,
                          'IMD_2010_national'].mean()
assert d1_score > d10_score, \
    f'Decile direction wrong: D1 mean={d1_score:.1f}, D10 mean={d10_score:.1f}'
print(f'\nDecile direction verified: D1 (most deprived, mean={d1_score:.1f}) > '
      f'D10 (least deprived, mean={d10_score:.1f})')



Decile direction verified: D1 (most deprived, mean=50.7) > D10 (least deprived, mean=5.7)


In [8]:
# ---- 2d. Summary: national decile composition ----
print(f'\nNational decile distribution:')
print(f'{"Decile":>6s} {"N_total":>8s} {"N_London":>9s} {"Mean IMD Score":>9s}')
for d in range(1, 11):
    mask = msoa_imd['Wealth_Decile_National'] == d
    n_total = mask.sum()
    n_london = (mask & msoa_imd['msoa11cd'].isin(london_msoas)).sum()
    mean_imd = msoa_imd.loc[mask, 'IMD_2010_national'].mean()
    print(f'  D{d:>2d}   {n_total:>7d}  {n_london:>8d}  {mean_imd:>9.2f}')



National decile distribution:
Decile  N_total  N_London Mean IMD Score
  D 1       678       107      50.73
  D 2       678       182      36.99
  D 3       678       149      29.63
  D 4       677       135      24.06
  D 5       678       107      19.72
  D 6       678        76      16.22
  D 7       677        74      13.41
  D 8       678        65      11.03
  D 9       678        52       8.74
  D10       678        36       5.73


> **Not the heavy tails on both ends as we expected? Specifically, there're more deprived MSOAs in London?**

In [9]:
# ---- 2e. External MSOA decile ----
# Population-weighted mean decile of all non-London MSOAs
non_london_imd = msoa_imd[~msoa_imd['msoa11cd'].isin(london_msoas)].copy()

# Get MSOA-level population for weighting
msoa_pop = (imd_with_msoa.groupby('msoa11cd')['pop'].sum()
            .reset_index(name='msoa_pop'))
non_london_imd = non_london_imd.merge(msoa_pop, on='msoa11cd', how='left')
non_london_imd['msoa_pop'] = non_london_imd['msoa_pop'].fillna(0)

# Safety check
total_ext_pop = non_london_imd['msoa_pop'].sum()
print(f'\nExternal MSOA calculation:')
print(f'  Non-London MSOAs: {len(non_london_imd)}, total pop: {total_ext_pop:,.0f}')
assert total_ext_pop > 0, 'External population is zero — check KS101 file coverage'

ext_weighted_decile = np.average(
    non_london_imd['Wealth_Decile_National'],
    weights=non_london_imd['msoa_pop'])
ext_decile = int(round(ext_weighted_decile))
print(f'  Weighted mean decile: {ext_weighted_decile:.2f} → assigned D{ext_decile}')


External MSOA calculation:
  Non-London MSOAs: 5795, total pop: 43,223,548
  Weighted mean decile: 5.67 → assigned D6


> **MSOAs of external flows are mainly in D6.**

In [10]:
# ---- 2f. Build wealth mapping (all MSOAs + external) ----
wealth_national = dict(zip(msoa_imd['msoa11cd'],
                            msoa_imd['Wealth_Decile_National']))
wealth_national[EXTERNAL_CODE] = ext_decile

In [11]:
# ---- 2g. Compare national vs London-relative deciles ----
print(f'\nLondon MSOA decile comparison (London-relative vs National):')
london_comparison = msoa_imd[msoa_imd['msoa11cd'].isin(london_msoas)].merge(
    existing[['msoa11cd', 'Wealth_Decile']], on='msoa11cd')

ct = pd.crosstab(london_comparison['Wealth_Decile'],
                 london_comparison['Wealth_Decile_National'], margins=True)
print(ct.to_string())

rho, p = stats.spearmanr(london_comparison['Wealth_Decile'],
                          london_comparison['Wealth_Decile_National'])
print(f'\nSpearman ρ = {rho:.4f}, p = {p:.2e}')


London MSOA decile comparison (London-relative vs National):
Wealth_Decile_National    1    2    3    4    5   6   7   8   9  10  All
Wealth_Decile                                                           
1                        99    0    0    0    0   0   0   0   0   0   99
2                         8   90    0    0    0   0   0   0   0   0   98
3                         0   92    6    0    0   0   0   0   0   0   98
4                         0    0   98    0    0   0   0   0   0   0   98
5                         0    0   45   53    0   0   0   0   0   0   98
6                         0    0    0   82   17   0   0   0   0   0   99
7                         0    0    0    0   90   8   0   0   0   0   98
8                         0    0    0    0    0  68  30   0   0   0   98
9                         0    0    0    0    0   0  44  54   0   0   98
10                        0    0    0    0    0   0   0  11  52  36   99
All                     107  182  149  135  107  76  74  65  5

### Interpretation

Rows are existing London-relative deciles.
Columns are national deciles.

- The striking pattern is the downward shift.
    - In the very bottom line, 107 MSOAs in D1, but only 36 in D10.
    - **London is more deprived than England on average. When ranking London MSOAs against the whole country, they pile up in the lower deciles.**

- London D1 all stay at national D1.
    -  These are the most deprived in the country, not just in London.

- London D5 splits into national D3 and national D4.
    - "Middle" within London is actually below-average nationally.

- London D10 only have 36 of 99 in national D10 as well.
    - The rest London D10 spread across national D7 and D9.
    - Even London's wealthiest neighbourhoods are not all in the national top 10%.

- rho = 0.99 means the ordering is almost identical.
    - e.g. If MSOA a is wealthier than MSOA b within London, then it should be wealthier natioanlly.
    - Even the fact of "wealthier" holding, the levels would compress downward.
    - **The internal hierarchy is preserved, but only the absolute position changes.**

External MSOAs are mostly at D6 nationally, sitting above roughly 75% of London MSOAs in national frame, since only 164 London MSOAs reach national D6 or higher.

So, **most flows from outside London into London would register as `Inflow_Wealthier` (cascade direction), and most flows from London to outside will register as `Outflow_Wealthier`.**

## 3. Filter OD data: London-touching flows

Keep flows where at least one endpoint is a Lonson MSOA.
Non-London endpoints are recoded as `EXTERNAL_CODE`.
We exclude intra-MSOA moves.